# Calibrate Gaia XP mean spectra

GaiaXPy provides the function `calibrate` to sample continuous XP mean spectra on an absolute wavelength and flux scale.

**Contents**

- [Input spectra](#gaiaxpy-input-spectra)
- [Default sampling](#gaiaxpy-default-sampling)
- [Custom sampling](#gaiaxpy-custom-sampling)
- [Spectrum plotting](#gaiaxpy-spectrum-plotting)
- [Additional options](#gaiaxpy-additional-options)
- [Compare with and without truncation](#gaiaxpy-truncation-comparison)
- [Output files](#gaiaxpy-output-files)

<a id="gaiaxpy-input-spectra" name="gaiaxpy-input-spectra"></a>

## Input spectra

The input consists of continuous XP mean spectra; this is BP/RP coefficients along with their errors and correlations. Local inputs can be CSV, ECSV, FITS, XML/VOTable, including supported gzip files and Archive ZIP downloads as obtained from the Gaia Archive. GaiaXPy also accepts a pandas DataFrame or a pandas Series. For a source-ID list or tuple, or ADQL query, GaiaXPy retrieves the spectra directly from the Archive.

**In order to run this tutorial, replace `spectra_input` with an existing file.**

In [ ]:
from gaiaxpy import calibrate, plot_spectra

spectra_input = "path/to/dr3_xp_mean_spectra.ecsv"

<a id="gaiaxpy-default-sampling" name="gaiaxpy-default-sampling"></a>

## Default sampling

The function returns a DataFrame and a sampling array in nm. The output contains `source_id`, `flux` and `flux_error`. The default grid or sampling covers 336–1020 nm, in 2 nm steps. Flux and flux error are in W nm⁻¹ m⁻². GaiaXPy combines BP and RP where both are available. For an input with data available only for a single passband, sampling values outside that passband's coverage are masked on the same grid. Both endpoints are included. Omitting `sampling` or passing `sampling=None` uses the default sampling.

In [ ]:
spectra, sampling = calibrate(spectra_input, save_file=False)
spectra.head()

# Inspect the first source at the first five positions.
import pandas as pd

first = spectra.iloc[0]
pd.DataFrame({"sampling_nm": sampling[:5], "flux": first["flux"][:5],
              "flux_error": first["flux_error"][:5]})

print(f"{sampling.size} sampling positions")

<a id="gaiaxpy-custom-sampling" name="gaiaxpy-custom-sampling"></a>

## Custom sampling

Custom sampling requires a NumPy array with at least two increasing sampling values in the interval [330, 1050] nm. The example evaluates the calibrated spectrum every 5 nm between 400 and 900 nm, including both endpoints. The returned sampling array belongs to `custom_spectra`.

In [ ]:
import numpy as np

grid = np.arange(400.0, 901.0, 5.0)
custom_spectra, custom_sampling = calibrate(spectra_input, sampling=grid, save_file=False)
custom_spectra.head()
print(custom_sampling[[0, -1]], custom_sampling.size)

<a id="gaiaxpy-spectrum-plotting" name="gaiaxpy-spectrum-plotting"></a>

## Spectrum plotting

The plot requires both the table and its sampling array. This example plots the first returned spectrum.

In [ ]:
from gaiaxpy import plot_spectra

plot_spectra(spectra.iloc[0], sampling)

# To plot multiple spectra on the same canvas, for example the first 5, use multi=True:
plot_spectra(spectra.iloc[:5], sampling, multi=True)

<a id="gaiaxpy-additional-options" name="gaiaxpy-additional-options"></a>

## Additional options

| Option | Default | Effect |
| --- | --- | --- |
| `truncation` | `False` | `True` selects the recommended number of basis functions from `bp_n_relevant_bases` and `rp_n_relevant_bases`. |
| `with_correlation` | `False` | `True` includes a column with the packed correlations between output samples. |
| `disable_info` | `False` | `True` suppresses routine messages and progress bars. |

<a id="gaiaxpy-truncation-comparison" name="gaiaxpy-truncation-comparison"></a>

## Compare with and without truncation

The default result above uses all coefficients. This example repeats the calculation using `truncation=True` and displays both results for the same source on the default sampling grid.

In [ ]:
import matplotlib.pyplot as plt

truncated_spectra, truncated_sampling = calibrate(spectra_input, truncation=True, save_file=False)
source_id = spectra["source_id"].iloc[0]
full_flux = spectra.loc[spectra["source_id"] == source_id, "flux"].iloc[0]
truncated_flux = truncated_spectra.loc[truncated_spectra["source_id"] == source_id, "flux"].iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sampling, full_flux, label="All coefficients")
ax.plot(truncated_sampling, truncated_flux, "--", label="Truncated")
ax.set(xlabel="Wavelength (nm)", ylabel=r"Flux (W nm$^{-1}$ m$^{-2}$)",
       title=f"Source {source_id}")
ax.legend()
fig.tight_layout()
plt.show()

<a id="gaiaxpy-output-files" name="gaiaxpy-output-files"></a>

## Output files

With `save_file=True`, GaiaXPy writes the results in the selected output format. This example writes an ECSV file (with metadata in the file itself).

```python
calibrate(spectra_input, save_file=True, output_file="calibrated_spectra", output_format="ecsv")
```

The default filename stem is `output_spectra`. `output_path` sets the destination directory. `output_format` accepts CSV, ECSV, FITS or XML/VOTable. If `output_format` is omitted, GaiaXPy uses the underlying input format, or CSV for pandas and Archive selections. CSV sampled-spectrum output writes separate `_sampling.csv` and `_metadata.csv` companions. The rest of the formats include the results along with the metadata and sampling in a single output file.